# XGBoost — Full Deep Dive

In [ ]:
The Story First 📖

Remember Random Forest?
→ 100 trees working TOGETHER
→ each tree independent
→ majority vote at end

XGBoost is different!
→ trees work SEQUENTIALLY
→ each tree LEARNS FROM
  previous tree's mistakes!

Like a student:
Random Forest = 100 students
               giving independent answers

XGBoost = 1 student
          attempting exam 100 times
          learning from each mistake! 🔥

# Boosting vs Bagging:

In [ ]:
BAGGING (Random Forest):
Tree 1 → prediction (independent)
Tree 2 → prediction (independent)
Tree 3 → prediction (independent)
Final  → majority vote

BOOSTING (XGBoost):
Tree 1 → prediction → finds mistakes
Tree 2 → focuses on Tree 1's mistakes
Tree 3 → focuses on Tree 2's mistakes
...
Final  → all trees combined! 🔥

Boosting is smarter!
Each tree improves on previous! ✅

# How XGBoost Works Internally:

In [ ]:
Step 1 → Tree 1 makes predictions
         Error = Actual - Predicted

Step 2 → Tree 2 tries to predict
         Tree 1's ERRORS

Step 3 → Tree 3 tries to predict
         Tree 2's remaining ERRORS

Step 4 → Final = Tree1 + Tree2 + Tree3...
         (weighted sum of all trees)

Example:
Actual marks = 90

Tree 1 predicts → 70  (error = 20)
Tree 2 predicts → 15  (error = 5)
Tree 3 predicts → 4   (error = 1)

Final = 70 + 15 + 4 = 89 ≈ 90 ✅
Getting closer each time!

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score,
                             confusion_matrix,
                             classification_report)
from xgboost import XGBClassifier, plot_importance

In [3]:
# BIG dataset — Loan Default Prediction
np.random.seed(42)
n = 500   # 500 customers!

data = {
    "age"              : np.random.randint(21, 65, n),
    "income"           : np.random.randint(20000, 200000, n),
    "loan_amount"      : np.random.randint(5000, 100000, n),
    "loan_tenure"      : np.random.randint(1, 30, n),
    "credit_score"     : np.random.randint(300, 850, n),
    "existing_loans"   : np.random.randint(0, 5, n),
    "employment_years" : np.random.randint(0, 20, n),
    "savings_amount"   : np.random.randint(0, 500000, n),
    "monthly_expenses" : np.random.randint(5000, 50000, n),
    "missed_payments"  : np.random.randint(0, 10, n),
    "education_level"  : np.random.randint(0, 4, n),
    "has_property"     : np.random.randint(0, 2, n),
}

# target → loan default or not
default = (
    (data["credit_score"] < 500)      |
    (data["missed_payments"] > 5)     |
    (data["existing_loans"] > 3)      &
    (data["income"] < 50000)          |
    (data["loan_amount"] > 80000)     &
    (data["savings_amount"] < 10000)
).astype(int)

In [6]:
data["default"]= default
df=pd.DataFrame(data)
print("Dataset Shape:", df.shape)
print("\nDefault Distribution:")
print(df["default"].value_counts())

Dataset Shape: (500, 13)

Default Distribution:
default
1    320
0    180
Name: count, dtype: int64


# Model Building: